<a href="https://colab.research.google.com/github/Ld01Rocha/AcelerometroSensor/blob/main/autogen_hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentes com Autogen - Function Calling - Hands-On

Agradecimento ao professor Julio Cesar dos Reis pelo material.

## Descrição

Exploramos a implementação de agentes simples utilizando a biblioteca Autogen. O objetivo é demonstrar como configurar e interagir com agentes capazes de executar tarefas automatizadas de forma eficiente. Aprenderemos a estruturar prompts, configurar agentes com diferentes especializações e testar interações dinâmicas entre eles.


In [ ]:
!pip install autogen
!pip install --upgrade pydantic
!pip install ipywidgets
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 834.0/834.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 5.6 MB/s eta 0:00:00


In [ ]:
from typing import Literal
import autogen
from typing_extensions import Annotated
import getpass

In [ ]:
# Solicitar a API key de forma segura
from google.colab import userdata
api_key =  userdata.get('groq')

config_list = [
    {
      "model": "llama-3.3-70b-versatile",  # "llama3-8b-8192",
      "api_key": api_key,
      "api_type": "groq"
    }
]

Uma ferramenta deve ser registrada com pelo menos dois agentes para que seja útil na conversação. O agente registrado com a assinatura da ferramenta por meio de register_for_llm pode chamar a ferramenta; o agente registrado com o objeto de função da ferramenta por meio de register_for_execution pode executar a função da ferramenta.

In [ ]:
llm_config = {
    "config_list": config_list,
    "timeout": 120
}

#agente que efetua conversão de moeda
currency_bot = autogen.AssistantAgent(
    name="currency_bot",
    system_message="Para tarefas relacionadas com conversão de moedas, use apenas as funções que você recebeu. Responda TERMINATE "
                   "quando a tarefa estiver concluída.",
    llm_config=llm_config
)

#agente user proxy
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
    code_execution_config=False
)

CurrencySymbol = Literal["USD", "EUR"]

#função de taxa_de_câmbio
def exchange_rate(base_currency: CurrencySymbol, quote_currency: CurrencySymbol) -> float:
    if base_currency == quote_currency:
        return 1.0
    elif base_currency == "USD" and quote_currency == "EUR":
        return 1 / 1.09
    elif base_currency == "EUR" and quote_currency == "USD":
        return 1 / 1.1
    else:
        raise ValueError(f"Moedas desconhecidas: {base_currency}, {quote_currency}")  # Tradução: "Moedas desconhecidas:..."

#registrando a ferramenta
@user_proxy.register_for_execution()
# Register the tool signature with the assistant agent.
@currency_bot.register_for_llm(description="Calculadora de câmbio de moeda")  # Tradução: "Calculadora de câmbio de moeda"
def currency_calculator(
        base_amount: Annotated[float, "Quantia da moeda em base_currency"],  # Tradução: "Quantia da moeda em base_currency"
        base_currency: Annotated[CurrencySymbol, "Moeda base"] = "USD",  # Tradução: "Moeda base"
        quote_currency: Annotated[CurrencySymbol, "Moeda de cotação"] = "EUR"  # Tradução: "Moeda de cotação"
) -> str:
    quote_amount = exchange_rate(base_currency, quote_currency) * base_amount
    return f"{quote_amount} - {quote_currency}"



#######################################
#inicialização da conversa
user_proxy.initiate_chat(
    currency_bot,
    message="E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?"  # Tradução: "Converter 100 USD para EUR"
    # message="E ai, truta! Está calor, hein? Estou na sala"

)


user_proxy (to currency_bot):

E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?

--------------------------------------------------------------------------------
currency_bot (to user_proxy):

***** Suggested tool call (fxwxdvc65): currency_calculator *****
Arguments: 
{"base_amount":200,"base_currency":"USD","quote_currency":"EUR"}
****************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION currency_calculator...
Call ID: fxwxdvc65
Input arguments: {'base_amount': 200, 'base_currency': 'USD', 'quote_currency': 'EUR'}

>>>>>>>> EXECUTED FUNCTION currency_calculator...
Call ID: fxwxdvc65
Input arguments: {'base_amount': 200, 'base_currency': 'USD', 'quote_currency': 'EUR'}
Output:
183.4862385321101 - EUR
user_proxy (to currency_bot):

***** Response from calling tool (fxwxdvc65) *****
183.4862385321101 - EUR
****************************

/usr/local/lib/python3.12/dist-packages/autogen/oai/groq.py:315: UserWarning: Cost calculation not available for model llama-3.3-70b-versatile
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)


currency_bot (to user_proxy):

***** Suggested tool call (gff8dj1p5): currency_calculator *****
Arguments: 
{"base_amount":200,"base_currency":"USD","quote_currency":"EUR"}
****************************************************************

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (90195bfc-081c-4d00-940c-073ca64d3284): Maximum number of consecutive auto-replies reached


ChatResult(chat_id=None, chat_history=[{'content': 'E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?', 'role': 'assistant', 'name': 'user_proxy'}, {'tool_calls': [{'id': 'fxwxdvc65', 'function': {'arguments': '{"base_amount":200,"base_currency":"USD","quote_currency":"EUR"}', 'name': 'currency_calculator'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}, {'content': '183.4862385321101 - EUR', 'tool_responses': [{'tool_call_id': 'fxwxdvc65', 'role': 'tool', 'content': '183.4862385321101 - EUR'}], 'role': 'tool', 'name': 'user_proxy'}, {'tool_calls': [{'id': 'gff8dj1p5', 'function': {'arguments': '{"base_amount":200,"base_currency":"USD","quote_currency":"EUR"}', 'name': 'currency_calculator'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}], summary='', cost={'usage_including_cached_inference': {'total_cost': 0.0, 'llama-3.3-70b-versatile': {'cost': 0.0, 'prompt_tokens': 743, 'completion_tokens': 72, 'total_tokens': 815}}, '

# Seu código vem aqui abaixo
$\color{red}{\text{Agora é sua vez, cria um agente com seu objetivo e suas ações}}$

In [ ]:
llm_config = {
    "config_list": config_list,
    "timeout": 120
}

#agente que efetua conversão de moeda
currency_bot = autogen.AssistantAgent(
    name="currency_bot",
    system_message="Para tarefas relacionadas com automação residencial, use apenas as funções que você recebeu. Responda TERMINATE "
                   "quando a tarefa estiver concluída.",
    llm_config=llm_config
)

#agente user proxy
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
    code_execution_config=False
)

EnviromentSymbol = Literal["Sala de estar", "quarto", "cozinha"]

#registrando a ferramenta
@user_proxy.register_for_execution()
# Register the tool signature with the assistant agent.
@currency_bot.register_for_llm(description="Liga ou desliga o ar condicionado de algum ambiente")
def turn_on_airconditioner(
        ambiente: Annotated[EnviromentSymbol, "Ambiente que é necessário ligar ou desligar o ar condicionado"],
        estado: Annotated[bool, "True para ligar o ar condicionado, False para desligar"]
) -> str:
    return f"Ar condicionado do {ambiente} foi {'ligado' if estado else 'desligado'}"


#######################################
#inicialização da conversa
user_proxy.initiate_chat(
    currency_bot,
    # message="E ai, pai? Tenho 200 dolares e estou indo pra europa, quanto eu tenho na moeda local?"  # Tradução: "Converter 100 USD para EUR"
    message="E ai, truta! Está calor, hein? Estou na sala"

)


user_proxy (to currency_bot):

E ai, truta! Está calor, hein? Estou na sala

--------------------------------------------------------------------------------
currency_bot (to user_proxy):

***** Suggested tool call (g2amfbqzg): turn_on_airconditioner *****
Arguments: 
{"ambiente":"Sala de estar","estado":true}
*******************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION turn_on_airconditioner...
Call ID: g2amfbqzg
Input arguments: {'ambiente': 'Sala de estar', 'estado': True}

>>>>>>>> EXECUTED FUNCTION turn_on_airconditioner...
Call ID: g2amfbqzg
Input arguments: {'ambiente': 'Sala de estar', 'estado': True}
Output:
Ar condicionado do Sala de estar foi ligado
user_proxy (to currency_bot):

***** Response from calling tool (g2amfbqzg) *****
Ar condicionado do Sala de estar foi ligado
**************************************************

-----------------------------------

ChatResult(chat_id=None, chat_history=[{'content': 'E ai, truta! Está calor, hein? Estou na sala', 'role': 'assistant', 'name': 'user_proxy'}, {'tool_calls': [{'id': 'g2amfbqzg', 'function': {'arguments': '{"ambiente":"Sala de estar","estado":true}', 'name': 'turn_on_airconditioner'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}, {'content': 'Ar condicionado do Sala de estar foi ligado', 'tool_responses': [{'tool_call_id': 'g2amfbqzg', 'role': 'tool', 'content': 'Ar condicionado do Sala de estar foi ligado'}], 'role': 'tool', 'name': 'user_proxy'}, {'content': 'Pronto! Estou sentado na minha cadeira, recebendo um ar fresco e uma brisa agradável.', 'role': 'user', 'name': 'currency_bot'}], summary='Pronto! Estou sentado na minha cadeira, recebendo um ar fresco e uma brisa agradável.', cost={'usage_including_cached_inference': {'total_cost': 0.00011244, 'llama3-8b-8192': {'cost': 0.00011244, 'prompt_tokens': 2084, 'completion_tokens': 103, 'total_tokens': 2187}}, 'usage_ex